# Phase 3 — Data Cleaning

Input  : data/raw/telco_churn_raw.csv
Output : data/cleaned/telco_churn_clean.csv

Every cleaning decision in this notebook is documented
with its business justification. No change is made
without a reason.

In [1]:
import pandas as pd
import numpy as np

# Fresh load every time this notebook runs
# This guarantees df_clean always starts from
# the original raw file — no stale state possible
df_raw = pd.read_csv('../data/raw/telco_churn_raw.csv')

# .copy() creates a fully independent duplicate
# Changes to df_clean will never affect df_raw
# df_raw stays as our permanent reference point
df_clean = df_raw.copy()

print(f"Raw file loaded successfully")
print(f"Shape : {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print(f"TotalCharges dtype on load : {df_clean['TotalCharges'].dtype}")

Raw file loaded successfully
Shape : 7043 rows, 21 columns
TotalCharges dtype on load : str


In [2]:
print("=" * 55)
print("COMPLETE CLEANING SEQUENCE")
print("=" * 55)

# ── FIX 1 ───────────────────────────────────────────────
# PROBLEM : TotalCharges is stored as text
# BUSINESS REASON : It represents money — must be numeric
#                   for all revenue calculations
# SOLUTION : Convert to float, blank strings become NaN

df_clean['TotalCharges'] = pd.to_numeric(
    df_clean['TotalCharges'], errors='coerce'
)

# Verify Fix 1 immediately
nan_count = df_clean['TotalCharges'].isnull().sum()
print(f"\nFix 1 — TotalCharges conversion")
print(f"  New dtype      : {df_clean['TotalCharges'].dtype}")
print(f"  NaN created    : {nan_count}")

# Defensive check — stop and warn if unexpected
if df_clean['TotalCharges'].dtype != 'float64':
    print("  WARNING: Conversion failed. Stop here.")
else:
    print("  Status: Fix 1 passed")

# ── FIX 2 ───────────────────────────────────────────────
# PROBLEM : 11 rows have NaN in TotalCharges
# BUSINESS REASON : These are tenure=0 customers with no
#                   behavioral history. They add no value
#                   to churn prediction and would create
#                   false patterns in the model.
# SOLUTION : Remove these 11 rows

rows_before = len(df_clean)

# dropna removes rows where the specified column is NaN
# subset limits removal to TotalCharges column only
# inplace=True applies change directly to df_clean
df_clean.dropna(subset=['TotalCharges'], inplace=True)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"\nFix 2 — Remove incomplete rows")
print(f"  Rows before    : {rows_before}")
print(f"  Rows after     : {rows_after}")
print(f"  Rows removed   : {rows_removed}")
print(f"  Data retained  : {round(rows_after/rows_before*100, 2)}%")

if rows_removed == 11:
    print("  Status: Fix 2 passed")
else:
    print(f"  WARNING: Expected 11, got {rows_removed}. Investigate.")

# ── FIX 3 ───────────────────────────────────────────────
# PROBLEM : Churn column contains text — 'Yes' and 'No'
# BUSINESS REASON : Machine learning models cannot process
#                   text labels. They need numbers.
#                   1 = churned, 0 = stayed
# SOLUTION : Map Yes→1 and No→0

# .map() replaces each value using a dictionary lookup
# {'Yes':1, 'No':0} is the mapping dictionary
# Every 'Yes' becomes 1, every 'No' becomes 0
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

churn_nulls = df_clean['Churn'].isnull().sum()
churn_counts = df_clean['Churn'].value_counts()

print(f"\nFix 3 — Churn column encoding")
print(f"  New dtype      : {df_clean['Churn'].dtype}")
print(f"  Null values    : {churn_nulls}")
print(f"  Value counts   : {churn_counts.to_dict()}")

if churn_nulls == 0:
    print("  Status: Fix 3 passed")
else:
    print("  WARNING: Unexpected nulls in Churn. Investigate.")

# ── FIX 4 ───────────────────────────────────────────────
# PROBLEM : After removing 11 rows the index has gaps
#           Index jumps from 487 to 489 for example
#           This causes subtle bugs in some operations
# BUSINESS REASON : Clean sequential index is best practice
#                   for any dataset used in ML pipelines
# SOLUTION : Reset index to clean 0,1,2,3... sequence

# reset_index() rebuilds the index from 0 to n-1
# drop=True means do not add the old index as a column
# inplace=True applies change directly to df_clean
df_clean.reset_index(drop=True, inplace=True)

print(f"\nFix 4 — Index reset")
print(f"  New index start : {df_clean.index[0]}")
print(f"  New index end   : {df_clean.index[-1]}")
print(f"  Status: Fix 4 passed")

COMPLETE CLEANING SEQUENCE

Fix 1 — TotalCharges conversion
  New dtype      : float64
  NaN created    : 11
  Status: Fix 1 passed

Fix 2 — Remove incomplete rows
  Rows before    : 7043
  Rows after     : 7032
  Rows removed   : 11
  Data retained  : 99.84%
  Status: Fix 2 passed

Fix 3 — Churn column encoding
  New dtype      : int64
  Null values    : 0
  Value counts   : {0: 5163, 1: 1869}
  Status: Fix 3 passed

Fix 4 — Index reset
  New index start : 0
  New index end   : 7031
  Status: Fix 4 passed


## Cleaning Decision — The 11 NaN Rows

These 11 customers have tenure = 0 and Churn = No.
They represent day-zero customers with no billing history.

Option A (fill with 0): Rejected.
Reason: Would teach the model a false pattern.
Zero-tenure customers show No churn not because
they are loyal but because they haven't had time
to churn. This misleads the prediction model.

Option B (remove rows): Selected.
Reason: These records have no predictive value for
churn analysis. Removing 11 of 7,043 rows is a
0.15% reduction with positive impact on model quality.

Industry principle: incomplete behavioral records
are removed, not filled, when building predictive systems.

In [3]:
print("=" * 55)
print("FINAL VERIFICATION")
print("=" * 55)

# Check 1 — correct shape
print(f"\nShape : {df_clean.shape}")
print(f"Expected : (7032, 21)")

# Check 2 — no remaining nulls anywhere
total_nulls = df_clean.isnull().sum().sum()
print(f"\nTotal nulls remaining : {total_nulls}")
print(f"Expected : 0")

# Check 3 — data types of key columns
print(f"\nKey column dtypes:")
print(f"  TotalCharges : {df_clean['TotalCharges'].dtype}")
print(f"  Churn        : {df_clean['Churn'].dtype}")
print(f"  tenure       : {df_clean['tenure'].dtype}")

# Check 4 — Churn only contains 0 and 1
print(f"\nChurn unique values : {sorted(df_clean['Churn'].unique())}")
print(f"Expected : [0, 1]")

# Check 5 — TotalCharges is now truly numeric
print(f"\nTotalCharges sample values:")
print(df_clean['TotalCharges'].head(5).tolist())

FINAL VERIFICATION

Shape : (7032, 21)
Expected : (7032, 21)

Total nulls remaining : 0
Expected : 0

Key column dtypes:
  TotalCharges : float64
  Churn        : int64
  tenure       : int64

Churn unique values : [np.int64(0), np.int64(1)]
Expected : [0, 1]

TotalCharges sample values:
[29.85, 1889.5, 108.15, 1840.75, 151.65]


In [4]:
print("=" * 55)
print("SAVING CLEAN DATASET")
print("=" * 55)

# Save to the cleaned folder — never the raw folder
# index=False means do not write the row numbers
# as a column in the CSV file — keeps it clean
output_path = '../data/cleaned/telco_churn_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Clean file saved to : {output_path}")
print(f"Rows saved          : {len(df_clean)}")
print(f"Columns saved       : {df_clean.shape[1]}")
print("\nData cleaning phase complete.")
print("All future analysis will use the cleaned file only.")

SAVING CLEAN DATASET
Clean file saved to : ../data/cleaned/telco_churn_clean.csv
Rows saved          : 7032
Columns saved       : 21

Data cleaning phase complete.
All future analysis will use the cleaned file only.


## Cleaning Phase Complete

Input  : telco_churn_raw.csv  — 7,043 rows, 21 columns
Output : telco_churn_clean.csv — 7,032 rows, 21 columns

Changes made:
1. TotalCharges converted from text to float64
2. 11 zero-tenure rows removed (no predictive value)
3. Churn encoded: Yes→1, No→0
4. Index reset to clean sequential 0–7031

Data quality status:
- Zero null values remaining
- All columns have correct data types
- Target variable (Churn) is numeric and binary
- Raw file untouched and preserved

This cleaned file is the single source of truth
for all analysis, SQL queries, visualisations,
and machine learning in this project.